In [ ]:
!pip install pyspark


In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Retail Sales Analytics") \
    .getOrCreate()

print("Spark Version:", spark.version)

Spark Version: 4.0.3


In [ ]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType, DateType

schema = StructType([
    StructField("Row ID", IntegerType(), True),
    StructField("Order ID", StringType(), True),
    StructField("Order Date", StringType(), True), # Keep as String for now, convert later
    StructField("Ship Date", StringType(), True),  # Keep as String for now, convert later
    StructField("Ship Mode", StringType(), True),
    StructField("Customer ID", StringType(), True),
    StructField("Customer Name", StringType(), True),
    StructField("Segment", StringType(), True),
    StructField("Country", StringType(), True),
    StructField("City", StringType(), True),
    StructField("State", StringType(), True),
    StructField("Postal Code", IntegerType(), True),
    StructField("Region", StringType(), True),
    StructField("Product ID", StringType(), True),
    StructField("Category", StringType(), True),
    StructField("Sub-Category", StringType(), True),
    StructField("Product Name", StringType(), True),
    StructField("Sales", StringType(), True), # Explicitly read as StringType
    StructField("Quantity", IntegerType(), True),
    StructField("Discount", DoubleType(), True),
    StructField("Profit", DoubleType(), True)
])

df = spark.read.csv("/content/Sample - Superstore.csv",
                    header=True,
                    schema=schema) # Use the defined schema

df.show(5)

+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+
|Row ID|      Order ID|Order Date| Ship Date|     Ship Mode|Customer ID|  Customer Name|  Segment|      Country|           City|     State|Postal Code|Region|     Product ID|       Category|Sub-Category|        Product Name|   Sales|Quantity|Discount|  Profit|
+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+
|     1|CA-2016-152156| 11/8/2016|11/11/2016|  Second Class|   CG-12520|    Claire Gute| Consumer|United States|      Henderson|  Kentucky|      42420| South|FUR-BO-10001798|      Furniture|   Bookcases|Bush Somerset 

In [ ]:
from pyspark.sql.functions import to_date, col, when, lit, rlike
from pyspark.sql.types import DoubleType, IntegerType, StringType

numeric_pattern = r"^\s*[-+]?((\d+(\.\d*)?)|(\.\d+))([eE][-+]?\d+)?\s*$"

# Step 1: Create a temporary column with cleaned sales strings (non-numeric become None)
df_temp = df.withColumn("_sales_clean_string",
                        when(col("Sales").rlike(numeric_pattern), col("Sales"))
                        .otherwise(lit(None).cast(StringType())))

# Step 2: Cast the cleaned string to DoubleType and replace the original Sales column
df = df_temp.withColumn("Sales", col("_sales_clean_string").cast(DoubleType())) \
           .drop("_sales_clean_string") # Drop the temporary column

# Apply other transformations
df = df.withColumn("Quantity", col("Quantity").cast(IntegerType())) \
       .withColumn("Discount", col("Discount").cast(DoubleType())) \
       .withColumn("Order Date", to_date(col("Order Date"), "M/d/yyyy")) \
       .withColumn("Ship Date", to_date(col("Ship Date"), "M/d/yyyy"))
df.printSchema()

root
 |-- Row ID: integer (nullable = true)
 |-- Order ID: string (nullable = true)
 |-- Order Date: date (nullable = true)
 |-- Ship Date: date (nullable = true)
 |-- Ship Mode: string (nullable = true)
 |-- Customer ID: string (nullable = true)
 |-- Customer Name: string (nullable = true)
 |-- Segment: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- City: string (nullable = true)
 |-- State: string (nullable = true)
 |-- Postal Code: integer (nullable = true)
 |-- Region: string (nullable = true)
 |-- Product ID: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Sub-Category: string (nullable = true)
 |-- Product Name: string (nullable = true)
 |-- Sales: double (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- Discount: double (nullable = true)
 |-- Profit: double (nullable = true)



In [ ]:
print("Rows :", df.count())
print("Columns :", len(df.columns))

Rows : 9994
Columns : 21


In [ ]:
from pyspark.sql.functions import col, sum

df.select([
    sum(col(c).isNull().cast("int")).alias(c)
    for c in df.columns
]).show(truncate=False)

+------+--------+----------+---------+---------+-----------+-------------+-------+-------+----+-----+-----------+------+----------+--------+------------+------------+-----+--------+--------+------+
|Row ID|Order ID|Order Date|Ship Date|Ship Mode|Customer ID|Customer Name|Segment|Country|City|State|Postal Code|Region|Product ID|Category|Sub-Category|Product Name|Sales|Quantity|Discount|Profit|
+------+--------+----------+---------+---------+-----------+-------------+-------+-------+----+-----+-----------+------+----------+--------+------------+------------+-----+--------+--------+------+
|0     |0       |0         |0        |0        |0          |0            |0      |0      |0   |0    |0          |0     |0         |0       |0           |0           |300  |300     |11      |0     |
+------+--------+----------+---------+---------+-----------+-------------+-------+-------+----+-----+-----------+------+----------+--------+------------+------------+-----+--------+--------+------+



In [ ]:
print("Duplicate Rows:",
      df.count() - df.dropDuplicates().count())

Duplicate Rows: 0


In [ ]:
df.show(10)

+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+
|Row ID|      Order ID|Order Date| Ship Date|     Ship Mode|Customer ID|  Customer Name|  Segment|      Country|           City|     State|Postal Code|Region|     Product ID|       Category|Sub-Category|        Product Name|   Sales|Quantity|Discount|  Profit|
+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+
|     1|CA-2016-152156|2016-11-08|2016-11-11|  Second Class|   CG-12520|    Claire Gute| Consumer|United States|      Henderson|  Kentucky|      42420| South|FUR-BO-10001798|      Furniture|   Bookcases|Bush Somerset 

In [ ]:
from pyspark.sql.functions import sum

df.select(sum("Sales").alias("Total Sales")).show()

+------------------+
|       Total Sales|
+------------------+
|2272449.8562999545|
+------------------+



In [ ]:
df.select(sum("Profit").alias("Total Profit")).show()

+------------------+
|      Total Profit|
+------------------+
|285707.60220000165|
+------------------+



In [ ]:
from pyspark.sql.functions import desc

df.groupBy("Category") \
  .sum("Sales") \
  .withColumnRenamed("sum(Sales)", "Total Sales") \
  .orderBy(desc("Total Sales")) \
  .show()

+---------------+-----------------+
|       Category|      Total Sales|
+---------------+-----------------+
|     Technology|835900.0669999964|
|      Furniture|733046.8612999996|
|Office Supplies|703502.9280000031|
+---------------+-----------------+



In [ ]:
df.groupBy("Product Name") \
  .sum("Sales") \
  .withColumnRenamed("sum(Sales)", "Sales") \
  .orderBy(desc("Sales")) \
  .show(10)

+--------------------+------------------+
|        Product Name|             Sales|
+--------------------+------------------+
|Canon imageCLASS ...|         61599.824|
|Fellowes PB500 El...|         27453.384|
|Cisco TelePresenc...|          22638.48|
|HON 5400 Series T...|         21870.576|
|GBC DocuBind TL30...|19823.479000000003|
|GBC Ibimaster 500...|           19024.5|
|Hewlett Packard L...|         18839.686|
|"HP Designjet T52...|         18374.895|
|GBC DocuBind P400...|         17965.068|
|High Speed Automa...|17030.311999999998|
+--------------------+------------------+
only showing top 10 rows


In [ ]:
df.groupBy("Region") \
  .sum("Profit") \
  .withColumnRenamed("sum(Profit)", "Profit") \
  .orderBy(desc("Profit")) \
  .show()

+-------+------------------+
| Region|            Profit|
+-------+------------------+
|   West|107303.70150000004|
|   East| 91603.05670000015|
|  South|46650.341000000044|
|Central| 40150.50299999999|
+-------+------------------+



In [ ]:
from pyspark.sql.functions import month, year, sum

df.groupBy(
    year("Order Date").alias("Year"),
    month("Order Date").alias("Month")
).agg(
    sum("Sales").alias("Total Sales")
).orderBy("Year", "Month").show()

+----+-----+------------------+
|Year|Month|       Total Sales|
+----+-----+------------------+
|2014|    1|14161.348999999998|
|2014|    2|          4119.816|
|2014|    3| 55526.19900000002|
|2014|    4|28139.560999999994|
|2014|    5|         23634.667|
|2014|    6| 34508.99560000003|
|2014|    7| 33500.87299999999|
|2014|    8| 27603.51249999999|
|2014|    9| 81496.80679999998|
|2014|   10|31394.940999999988|
|2014|   11| 78297.24069999997|
|2014|   12| 69379.83649999999|
|2015|    1|18085.115599999994|
|2015|    2|11924.271999999999|
|2015|    3|38621.291999999994|
|2015|    4|32640.482499999987|
|2015|    5|29325.970500000003|
|2015|    6|24659.684000000005|
|2015|    7| 28524.52099999999|
|2015|    8|36380.928199999995|
+----+-----+------------------+
only showing top 20 rows


In [ ]:
from pyspark.sql.functions import desc, sum

df.groupBy("Customer Name") \
  .agg(sum("Sales").alias("Total Sales")) \
  .orderBy(desc("Total Sales")) \
  .show(10)

+------------------+------------------+
|     Customer Name|       Total Sales|
+------------------+------------------+
|       Sean Miller|          25043.05|
|      Tamara Chand|19017.847999999998|
|      Raymond Buch|         15117.339|
|      Tom Ashbrook|          14595.62|
|     Adrian Barton|14355.610999999997|
|      Sanjit Chand|14142.333999999999|
|      Ken Lonsdale|         14071.917|
|      Hunter Lopez|12873.297999999999|
|      Sanjit Engle|12209.438000000002|
|Christopher Conant|         12129.072|
+------------------+------------------+
only showing top 10 rows


In [ ]:
from pyspark.sql.functions import avg

df.groupBy("Category") \
  .agg(avg("Discount").alias("Average Discount")) \
  .show()

+---------------+-------------------+
|       Category|   Average Discount|
+---------------+-------------------+
|Office Supplies|  0.392942719574964|
|      Furniture|0.23932323710364325|
|     Technology|0.15062263129398756|
+---------------+-------------------+



In [ ]:
from pyspark.sql.functions import sum, desc

df.groupBy("Product Name") \
  .agg(sum("Profit").alias("Total Profit")) \
  .orderBy(desc("Total Profit")) \
  .show(10)

+--------------------+------------------+
|        Product Name|      Total Profit|
+--------------------+------------------+
|Canon imageCLASS ...|25199.928000000004|
|Fellowes PB500 El...|          7753.039|
|Hewlett Packard L...|         6983.8836|
|Canon PC1060 Pers...|         4570.9347|
|"HP Designjet T52...|         4094.9766|
|Ativa V4110MDD Mi...|         3772.9461|
|3D Systems Cube P...|3717.9714000000004|
|Plantronics Savi ...|3696.2819999999997|
|Ibico EPK-21 Elec...|         3345.2823|
|Zebra ZM400 Therm...|          3343.536|
+--------------------+------------------+
only showing top 10 rows


In [ ]:
from pyspark.sql.functions import sum, desc

df.groupBy("Ship Mode") \
  .agg(sum("Sales").alias("Total Sales")) \
  .orderBy(desc("Total Sales")) \
  .show()

+--------------+------------------+
|     Ship Mode|       Total Sales|
+--------------+------------------+
|Standard Class|1342260.1939999827|
|  Second Class|       453341.8504|
|   First Class|       349494.8469|
|      Same Day|127352.96500000001|
+--------------+------------------+



In [ ]:
from pyspark.sql.functions import sum, desc

df.groupBy("State") \
  .agg(sum("Sales").alias("Total Sales")) \
  .orderBy(desc("Total Sales")) \
  .show(10)

+------------+------------------+
|       State|       Total Sales|
+------------+------------------+
|  California| 450567.5915000007|
|    New York|309453.63299999974|
|       Texas|169553.63180000003|
|  Washington|136590.17199999996|
|Pennsylvania|114911.23700000002|
|     Florida|         88876.883|
|    Illinois|  79009.2869999999|
|        Ohio| 76617.64499999993|
|    Michigan| 75991.58400000002|
|    Virginia| 70309.08999999998|
+------------+------------------+
only showing top 10 rows


In [ ]:
from pyspark.sql.functions import sum, desc

df.groupBy("City") \
  .agg(sum("Profit").alias("Total Profit")) \
  .orderBy(desc("Total Profit")) \
  .show(10)

+-------------+------------------+
|         City|      Total Profit|
+-------------+------------------+
|New York City|        62061.2982|
|  Los Angeles|30404.695899999973|
|      Seattle|28894.395700000005|
|San Francisco|17179.468499999988|
|      Detroit|13117.051600000003|
|    Lafayette|        10001.5463|
|      Jackson|         7552.6813|
|      Atlanta|         6993.6629|
|  Minneapolis| 6824.584599999999|
|     Columbus| 6380.506399999996|
+-------------+------------------+
only showing top 10 rows


In [ ]:
from pyspark.sql.functions import sum, desc

df.groupBy("Customer Name") \
  .agg(sum("Profit").alias("Total Profit")) \
  .orderBy(desc("Total Profit")) \
  .show(10)

+--------------------+------------------+
|       Customer Name|      Total Profit|
+--------------------+------------------+
|        Tamara Chand| 8964.482600000001|
|        Raymond Buch|         6976.0959|
|        Sanjit Chand| 5757.411899999999|
|        Hunter Lopez|5622.4292000000005|
|       Adrian Barton|         5438.9075|
|        Tom Ashbrook| 4703.788299999999|
|Christopher Martinez|3899.8903999999998|
|       Keith Dawkins|         3038.6254|
|         Andy Reiter|2884.6207999999997|
|       Daniel Raglin|2869.0760000000005|
+--------------------+------------------+
only showing top 10 rows


In [ ]:
from pyspark.sql.functions import sum, desc

df.groupBy("Segment") \
  .agg(sum("Sales").alias("Total Sales")) \
  .orderBy(desc("Total Sales")) \
  .show()

+-----------+------------------+
|    Segment|       Total Sales|
+-----------+------------------+
|   Consumer|1150166.1819999903|
|  Corporate| 696604.5138000002|
|Home Office| 425679.1605000003|
+-----------+------------------+



In [ ]:
from pyspark.sql.functions import sum, desc

df.groupBy("Category") \
  .agg(sum("Profit").alias("Total Profit")) \
  .orderBy(desc("Total Profit")) \
  .show()

+---------------+------------------+
|       Category|      Total Profit|
+---------------+------------------+
|     Technology|145388.29659999989|
|Office Supplies|120632.87839999991|
|      Furniture| 19686.42720000003|
+---------------+------------------+



In [ ]:
from pyspark.sql.functions import sum, desc

df.groupBy("Sub-Category") \
  .agg(sum("Sales").alias("Total Sales")) \
  .orderBy(desc("Total Sales")) \
  .show(10)

+------------+------------------+
|Sub-Category|       Total Sales|
+------------+------------------+
|      Phones| 329753.0880000001|
|      Chairs|328449.10300000076|
|     Storage|216803.21200000012|
|      Tables| 206965.5320000001|
|     Binders|199905.71700000006|
|    Machines|189238.63099999996|
| Accessories| 167380.3180000001|
|     Copiers|149528.02999999994|
|   Bookcases|114879.99629999997|
|  Appliances|        107532.161|
+------------+------------------+
only showing top 10 rows


In [ ]:
from pyspark.sql.functions import avg

df.groupBy("Category") \
  .agg(avg("Profit").alias("Average Profit")) \
  .show()

+---------------+------------------+
|       Category|    Average Profit|
+---------------+------------------+
|Office Supplies|20.018731895121128|
|      Furniture| 9.281672418670453|
|     Technology| 78.71591586356247|
+---------------+------------------+



In [ ]:
from pyspark.sql.functions import year, month, sum

df.groupBy(
    year("Order Date").alias("Year"),
    month("Order Date").alias("Month")
).agg(
    sum("Profit").alias("Total Profit")
).orderBy("Year", "Month").show()

+----+-----+------------------+
|Year|Month|      Total Profit|
+----+-----+------------------+
|2014|    1|         2417.0891|
|2014|    2| 786.0427999999997|
|2014|    3| 419.4309999999997|
|2014|    4|         3453.5602|
|2014|    5|         2732.5806|
|2014|    6|5058.2393999999995|
|2014|    7|-925.1018000000008|
|2014|    8|5353.9983999999995|
|2014|    9| 8215.784200000002|
|2014|   10| 3453.919199999999|
|2014|   11| 9182.182499999994|
|2014|   12| 8913.100699999999|
|2015|    1|         -3290.815|
|2015|    2|2807.7147000000004|
|2015|    3| 9684.061800000003|
|2015|    4|         4495.3873|
|2015|    5|         4553.4094|
|2015|    6|           3374.89|
|2015|    7|3189.0990000000006|
|2015|    8|         5305.8516|
+----+-----+------------------+
only showing top 20 rows


In [ ]:
from pyspark.sql.functions import sum, desc

df.groupBy("Product Name") \
  .agg(sum("Quantity").alias("Total Quantity")) \
  .orderBy(desc("Total Quantity")) \
  .show(10)

+--------------------+--------------+
|        Product Name|Total Quantity|
+--------------------+--------------+
|             Staples|           215|
|     Staple envelope|           170|
|   Easy-staple paper|           150|
|Staples in misc. ...|            86|
|KI Adjustable-Hei...|            74|
|Avery Non-Stick B...|            71|
|Storex Dura Pro B...|            71|
|GBC Premium Trans...|            67|
|Situations Contou...|            64|
|Staple-based wall...|            62|
+--------------------+--------------+
only showing top 10 rows


In [ ]:
from pyspark.sql.functions import avg

df.groupBy("Region") \
  .agg(avg("Sales").alias("Average Sales")) \
  .show()

+-------+------------------+
| Region|     Average Sales|
+-------+------------------+
|  South|246.34805889803695|
|Central| 220.2658729203543|
|   East|243.90205152394708|
|   West| 230.2263131655374|
+-------+------------------+

